# Validation: Synthetic vs. Actual Localization Metrics

This notebook validates fitted model parameters by:
1. Loading fitted parameters for each participant and interpolation method
   (`results/fitting_results.csv`, from notebook 07)
2. Recomputing the NLL with 500 Monte Carlo samples (`nll_500`) for model comparison (BIC; Fig. 6, Table 2)
3. Generating synthetic localization data (200 reps per target position) from the fitted parameters
4. Computing localization metrics for both synthetic and actual data and comparing them
   (posterior predictive checks; Table 1 and Fig. 5)

Every expensive step is cached in `results/` (`nll_500` column of the fitting results,
`actual_behavioural_metrics.csv`, `synthetic_vs_actual_comparison.csv`) and skipped when the cache
exists, so the notebook runs from the cached results without the HRTFs or the behavioural data.
Figures 5 and 6 are drawn by `figures/fig_metrics_actual_vs_synthetic.py` and
`figures/fig_bic_comparison.py`.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from scipy import stats
import warnings
import pyfar as pf
warnings.filterwarnings('ignore')

from bayesian_listener import BayesianListener
from bayesian_listener import metrics

import paths

print("✓ Imports complete")

✓ Imports complete


## Load Data

In [2]:
# Data paths
data_folder = str(paths.HRTF_DIR)            # participant SOFA files (data/hrtf/)
fitting_results_file = paths.FITTING_RESULTS  # results/fitting_results.csv

# Load fitted parameters
fitted_params_df = pd.read_csv(fitting_results_file)
fitted_params_df = fitted_params_df[fitted_params_df['success'] == True]
print(f"Loaded fitted parameters: {len(fitted_params_df)} successful fits")

if paths.BEHAV_FILE.exists():
    # Load target directions
    dirs = pd.read_csv(paths.DIRS_FILE)
    print(f"Target directions: {len(dirs)} locations")

    # Create target coordinates (pyfar)
    targets_coords = pf.Coordinates.from_spherical_elevation(
        np.deg2rad(dirs['azi_target'].values),
        np.deg2rad(dirs['ele_target'].values),
        np.ones(len(dirs)))

    # Load behavioral responses
    obs_tbl = pd.read_csv(paths.BEHAV_FILE)
    obs_tbl = obs_tbl[(obs_tbl['condition'] == 'Measured')]
    obs_tbl = obs_tbl[(obs_tbl['participant'] != 'P0247')]
    trial_counts = obs_tbl.groupby('participant').size()
    participants = obs_tbl['participant'].unique()
else:
    # cached results only (the behavioural data are needed only to recompute the caches)
    print("Behavioural data not found: running from the cached results in results/")
    obs_tbl = None
    participants = fitted_params_df['participant'].unique()

print(f'participants: {len(participants)}')
print(participants)

Loaded fitted parameters: 132 successful fits
Target directions: 33 locations
participants: 33
<StringArray>
['P0001', 'P0006', 'P0019', 'P0031', 'P0043', 'P0124', 'P0172', 'P0181',
 'P0015', 'P0107', 'P0155', 'P0204', 'P0213', 'P0224', 'P0225', 'P0228',
 'P0239', 'P0243', 'P0244', 'P0250', 'P0003', 'P0259', 'P0261', 'P0005',
 'P0020', 'P0238', 'P0271', 'P0330', 'P0347', 'P0348', 'P0349', 'P0350',
 'P0351']
Length: 33, dtype: str


In [3]:
# Recompute NLL with 500 repetitions for all fitted parameters (in parallel)
# Skip if nll_500 is already present in fitting_results
from joblib import Parallel, delayed, cpu_count
from functools import partial
from bayesian_listener.fitting import negloglik

# Check if nll_500 already exists and has values
if 'nll_500' in fitted_params_df.columns and fitted_params_df['nll_500'].notna().all():
    print("✓ nll_500 already computed, skipping recomputation")
    print(f"Mean nll_500: {fitted_params_df['nll_500'].mean():.2f}")
else:
    print("Computing nll_500 (not present or incomplete)...")

    def compute_nll_for_row(row_data, data_folder, obs_tbl_dict, targets_coords):
        """Compute NLL for a single participant-method combination."""
        participant_id = row_data['participant']
        method = row_data['method']

        try:
            # Load HRTF and prepare model
            subj_sofa_path = f'{data_folder}/{participant_id}_FreeFieldCompMinPhase_48kHz.sofa'
            model = BayesianListener(subj_sofa_path)
            model.compute_template(interpolation=method)

            # Get behavioral data for this participant
            subj_data = obs_tbl_dict[participant_id]

            # Build response coordinates (pyfar)
            response_coords = pf.Coordinates.from_spherical_elevation(
                np.deg2rad(subj_data['azi_response'].values),
                np.deg2rad(subj_data['ele_response'].values),
                np.ones(len(subj_data))
            )

            # Get target indices for each response
            response_targets = pf.Coordinates.from_spherical_elevation(
                np.deg2rad(subj_data['azi_target'].values),
                np.deg2rad(subj_data['ele_target'].values),
                np.ones(len(subj_data))
            )
            resp_targets_idx = targets_coords.find_nearest(response_targets)[0][0]

            # Extract target features from model
            target_indices_in_model = model.target.coords.find_nearest(targets_coords)[0][0]
            targets = model.target[target_indices_in_model]

            # Convert parameters to log space as expected by negloglik
            # negloglik expects: [log(sigma_ild), log(sigma_spectral), log(kappa_motor), log(tau_prior)]
            # where tau_prior = 1 / sigma_prior^2
            tau_prior = 1.0 / (row_data['sigma_prior'] ** 2)
            sigmas_log = np.array([
                np.log(row_data['sigma_ild']),
                np.log(row_data['sigma_spectral']),
                np.log(row_data['kappa_motor']),
                np.log(tau_prior)
            ])

            # Compute NLL with 500 repetitions
            nll_value = negloglik(
                model,
                targets,
                response_coords,
                resp_targets_idx,
                sigmas_log,
                num_repetitions=500
            )

            return {
                'participant': participant_id,
                'method': method,
                'nll_500': nll_value,
                'nll_original': row_data['nll'],
                'nll_diff': nll_value - row_data['nll'],
                'success': True
            }
        except Exception as e:
            print(f"Error processing {participant_id} - {method}: {str(e)}")
            return {
                'participant': participant_id,
                'method': method,
                'nll_500': np.nan,
                'nll_original': row_data['nll'],
                'nll_diff': np.nan,
                'success': False
            }

    print("Recomputing NLL with 500 repetitions (parallel processing)...")
    print("="*80)

    # Prepare data for parallel processing
    obs_tbl_dict = {pid: obs_tbl[obs_tbl['participant'] == pid] for pid in participants}
    row_dicts = fitted_params_df.to_dict('records')

    # Determine number of workers
    n_workers = min(cpu_count(), len(row_dicts))
    print(f"Using {n_workers} parallel workers for {len(row_dicts)} tasks")

    # Run parallel processing with joblib (works better in notebooks than multiprocessing)
    nll_results = Parallel(n_jobs=n_workers, verbose=10)(
        delayed(compute_nll_for_row)(row, data_folder, obs_tbl_dict, targets_coords)
        for row in row_dicts
    )

    # Convert to DataFrame
    nll_comparison_df = pd.DataFrame(nll_results)

    # Filter successful results
    successful_results = nll_comparison_df[nll_comparison_df['success'] == True]

    print("\n" + "="*80)
    print("NLL Comparison Summary:")
    print(successful_results.groupby('method').agg({
        'nll_original': ['mean', 'std'],
        'nll_500': ['mean', 'std'],
        'nll_diff': ['mean', 'std']
    }).round(2))

    # Add nll_500 to fitted_params_df
    fitted_params_df = fitted_params_df.merge(
        nll_comparison_df[['participant', 'method', 'nll_500']],
        on=['participant', 'method'],
        how='left'
    )

    fitted_params_df.to_csv(fitting_results_file, index=False)

    print("\n✓ NLL recomputation complete")
    print(f"Successful: {nll_comparison_df['success'].sum()}/{len(nll_comparison_df)}")

✓ nll_500 already computed, skipping recomputation
Mean nll_500: 233.80


In [4]:
# BIC Analysis: Mean ± SE plot
# Compute BIC (assumes n_obs already in fitted_params_df)
if 'n_obs' not in fitted_params_df.columns:
    if obs_tbl is not None:
        n_obs = obs_tbl.groupby('participant').size()
        fitted_params_df = fitted_params_df.merge(n_obs.rename('n_obs').reset_index(), on='participant', how='left')
    else:
        # n_trials (from fit_listener) is the same per-participant trial count
        fitted_params_df['n_obs'] = fitted_params_df['n_trials']

k_params = 3
fitted_params_df['BIC'] = k_params * np.log(fitted_params_df['n_obs']) + 2 * fitted_params_df['nll_500']


In [5]:
summary = fitted_params_df.groupby('method')[
    ['sigma_motor', 'sigma_spectral',  'sigma_prior', 'BIC']
].agg(['median', lambda x: np.percentile(x, q = 25), lambda x: np.percentile(x, q = 75)])

print(summary)

              sigma_motor                       sigma_spectral             \
                   median <lambda_0> <lambda_1>         median <lambda_0>   
method                                                                      
SH               9.781543   6.275714  16.587337       9.567274   7.685660   
SHMAX            9.637088   6.338223  16.469232       8.956590   7.641403   
barumerli2023    9.687607   6.297764  16.943001       8.817450   6.892534   
barycentric      9.681690   6.342839  16.429515       9.098805   7.564768   

                         sigma_prior                                BIC  \
              <lambda_1>      median <lambda_0>  <lambda_1>      median   
method                                                                    
SH             11.098133   50.528864  40.335310  129.859313  356.933850   
SHMAX          10.567750   50.000000  32.791447   88.753232  357.678819   
barumerli2023  10.000000   50.000000  29.934246   73.535122  339.792258   
barycentri

In [6]:
summary = fitted_params_df.groupby('method')[
    ['sigma_motor', 'sigma_spectral',  'sigma_prior', 'BIC']
].agg(['mean', 'sem'])

print(summary.to_latex(float_format = '%.2f'))

\begin{tabular}{lrrrrrrrr}
\toprule
 & \multicolumn{2}{r}{sigma_motor} & \multicolumn{2}{r}{sigma_spectral} & \multicolumn{2}{r}{sigma_prior} & \multicolumn{2}{r}{BIC} \\
 & mean & sem & mean & sem & mean & sem & mean & sem \\
method &  &  &  &  &  &  &  &  \\
\midrule
SH & 12.54 & 1.46 & 10.12 & 0.75 & 78.12 & 9.64 & 488.55 & 64.28 \\
SHMAX & 12.52 & 1.46 & 10.35 & 0.91 & 68.96 & 9.01 & 471.97 & 58.70 \\
barumerli2023 & 12.53 & 1.43 & 9.53 & 0.71 & 55.03 & 6.21 & 491.33 & 59.66 \\
barycentric & 12.51 & 1.46 & 9.50 & 0.56 & 73.49 & 9.99 & 478.78 & 64.19 \\
\bottomrule
\end{tabular}



In [7]:
# Calculate delta BIC relative to minimum BIC for each participant (Fig. 6)
bic_comparison = []
for pid in fitted_params_df['participant'].unique():
    p_data = fitted_params_df[fitted_params_df['participant'] == pid]
    min_bic = p_data['BIC'].min()
    for _, row in p_data.iterrows():
        bic_comparison.append({'participant': pid, 'method': row['method'], 'BIC': row['BIC'],
                              'delta_bic': row['BIC'] - min_bic})

bic_df = pd.DataFrame(bic_comparison)

# Mean and SE (plotted in Fig. 6 by figures/fig_bic_comparison.py)
stats_list = []
for method in ['barumerli2023', 'SHMAX', 'barycentric', 'SH']:
    data = bic_df[bic_df['method'] == method]['delta_bic'].values
    stats_list.append({'method': method, 'mean': np.mean(data),
                      'se': np.std(data) / np.sqrt(len(data))})
stats_df = pd.DataFrame(stats_list)
print('ΔBIC relative to the best method per participant (mean ± SE):')
print(stats_df.round(1).to_string(index=False))

ΔBIC relative to the best method per participant (mean ± SE):
       method  mean   se
barumerli2023  45.2  9.2
        SHMAX  25.8  5.8
  barycentric  32.6 12.3
           SH  42.4 12.0


In [8]:
# ΔBIC relative to barumerli2023 (reference method)
# Positive = better than reference, negative = worse
summary_stats = []
for method in ['SHMAX', 'SH', 'barycentric', 'barumerli2023']:
    delta_data = []
    for pid in fitted_params_df['participant'].unique():
        p_data = fitted_params_df[fitted_params_df['participant'] == pid]
        bic_ref = p_data[p_data['method'] == 'barumerli2023']['BIC'].values[0]
        bic_method = p_data[p_data['method'] == method]['BIC'].values[0]
        # Positive delta = reference is worse = this method wins
        delta_data.append(bic_method - bic_ref)

    delta_data = (np.array(delta_data))

    # Kass & Raftery: how many participants show evidence FOR this method over reference
    very_strong_for = int(np.sum(delta_data < -10))
    strong_for =  int(np.sum((delta_data < -6)  & (delta_data >= -10)))
    positive = int(np.sum((delta_data < -2)  & (delta_data >= -6)))

    wins = int(np.sum(delta_data <= -2))

    summary_stats.append({
        'Method': method,
        'Wins': wins,
        r'$\Delta$BIC': f"{np.median(delta_data):.1f}",
        r'Very Strong': very_strong_for,
        r'Strong': strong_for,
        r'Positive': positive
    })

latex_df = pd.DataFrame(summary_stats)
print(latex_df)
print(f"\nRow sums (excl. barumerli2023): "
      f"{[row[r'Very Strong'] + row[r'Strong'] + row[r'Positive'] for _, row in latex_df.iterrows()]}")
print("\n" + latex_df.to_latex(index=False, escape=False))

          Method  Wins $\Delta$BIC  Very Strong  Strong  Positive
0          SHMAX    19       -10.4           17       1         1
1             SH    15         2.4           13       0         2
2    barycentric    18        -4.8           15       1         2
3  barumerli2023     0         0.0            0       0         0

Row sums (excl. barumerli2023): [19, 15, 18, 0]

\begin{tabular}{lrlrrr}
\toprule
Method & Wins & $\Delta$BIC & Very Strong & Strong & Positive \\
\midrule
SHMAX & 19 & -10.4 & 17 & 1 & 1 \\
SH & 15 & 2.4 & 13 & 0 & 2 \\
barycentric & 18 & -4.8 & 15 & 1 & 2 \\
barumerli2023 & 0 & 0.0 & 0 & 0 & 0 \\
\bottomrule
\end{tabular}



In [9]:
summary_stats = []
for method in ['SHMAX', 'SH', 'barycentric', 'barumerli2023']:
    delta_data = []
    for pid in fitted_params_df['participant'].unique():
        p_data = fitted_params_df[fitted_params_df['participant'] == pid]
        bic_ref    = p_data[p_data['method'] == 'barumerli2023']['BIC'].values[0]
        bic_method = p_data[p_data['method'] == method]['BIC'].values[0]
        # ΔBIC = BIC_method - BIC_ref: negative = method is better
        delta_data.append(bic_method - bic_ref)

    delta_data = np.array(delta_data)

    # Evidence FOR this method over reference: delta < 0
    # |delta| ≈ 2*log(B), so thresholds follow Kass & Raftery on the 2logBF scale
    very_strong_for  = int(np.sum(delta_data < -10))
    strong_for       = int(np.sum((delta_data < -6)  & (delta_data >= -10)))
    positive_for     = int(np.sum((delta_data < -2)  & (delta_data >= -6)))
    negligible       = int(np.sum(np.abs(delta_data) <= 2))

    summary_stats.append({
        'Method': method,
        r'$\Delta$BIC (mean)': f"{np.mean(delta_data):.1f}$\pm${np.std(delta_data)/np.sqrt(33):.1f}",
        r'Very strong': very_strong_for,
        r'Strong':      strong_for,
        r'Positive':    positive_for,
        r'Negligible':  negligible,
    })


latex_df = pd.DataFrame(summary_stats)
print(latex_df)
print(f"\nRow sums (excl. barumerli2023): "
      f"{[row[r'Strong'] + row[r'Positive'] + row['Negligible'] for _, row in latex_df.iterrows()]}")
print("\n" + latex_df.to_latex(index=False, escape=False))

          Method $\Delta$BIC (mean)  Very strong  Strong  Positive  Negligible
0          SHMAX     -19.4$\pm$10.1           17       1         1           4
1             SH      -2.8$\pm$15.3           13       0         2           1
2    barycentric     -12.5$\pm$16.7           15       1         2           4
3  barumerli2023        0.0$\pm$0.0            0       0         0          33


Row sums (excl. barumerli2023): [6, 3, 7, 33]

\begin{tabular}{llrrrr}
\toprule
Method & $\Delta$BIC (mean) & Very strong & Strong & Positive & Negligible \\
\midrule
SHMAX & -19.4$\pm$10.1 & 17 & 1 & 1 & 4 \\
SH & -2.8$\pm$15.3 & 13 & 0 & 2 & 1 \\
barycentric & -12.5$\pm$16.7 & 15 & 1 & 2 & 4 \\
barumerli2023 & 0.0$\pm$0.0 & 0 & 0 & 0 & 33 \\
\bottomrule
\end{tabular}



## behavioral metrics

In [10]:
# Compute all metrics for actual behavioral data
# Skip if CSV already exists
behavioral_metrics_csv = paths.RESULTS_DIR / 'actual_behavioural_metrics.csv'

if Path(behavioral_metrics_csv).exists():
    print(f"✓ Loading existing behavioral metrics from {behavioral_metrics_csv.relative_to(paths.ROOT)}")
    actual_behavioral_df = pd.read_csv(behavioral_metrics_csv)
    print(f"Loaded {len(actual_behavioral_df)} participants")
else:
    print(f"Computing behavioral metrics (CSV not found)...")

    import bayesian_listener.metrics as metrics

    print(f"Available metrics: {list(metrics.METRIC_FUNCTIONS.keys())}")

    targets_hp = targets_coords.spherical_side    # (n_targets, 3) radians
    targets_sph = targets_coords.spherical_elevation  # (n_targets, 3) radians

    actual_metrics_rows = []

    for participant_id in participants:
        subj_data = obs_tbl[obs_tbl['participant'] == participant_id]

        # Convert responses to pyfar Coordinates
        resp_coords = pf.Coordinates.from_spherical_elevation(
            np.deg2rad(subj_data['azi_response'].values),
            np.deg2rad(subj_data['ele_response'].values),
            np.ones(len(subj_data))
        )
        resp_sph = resp_coords.spherical_elevation  # radians
        resp_hp = resp_coords.spherical_side        # radians

        # Get target indices
        resp_targets = pf.Coordinates.from_spherical_elevation(
            np.deg2rad(subj_data['azi_target'].values),
            np.deg2rad(subj_data['ele_target'].values),
            np.ones(len(subj_data))
        )
        target_indices = targets_coords.find_nearest(resp_targets)[0][0]

        # Per-response target coordinates (numpy arrays, radians)
        targ_hp = targets_hp[target_indices, :]
        targ_sph = targets_sph[target_indices, :]

        # Compute all registered metrics
        row = {'participant': participant_id}

        # Compute metrics at different cutoffs (in radians)
        for cutoff in [30, 60, 90]:
            for name in ['accL_cutoff', 'accP_cutoff']:
                func = metrics.METRIC_FUNCTIONS[name]
                meta = func._metadata
                if meta['coord_convention'] == 'horizontal-polar':
                    val, aux = func(targ_hp, resp_hp, cutoff=np.deg2rad(cutoff))
                elif meta['coord_convention'] == 'spherical':
                    val, aux = func(targ_sph, resp_sph, cutoff=np.deg2rad(cutoff))
                unit = meta.get('output_unit', '')
                if unit == 'radians':
                    row[f'{name}_{cutoff}_deg'] = np.rad2deg(val)
                else:
                    row[f'{name}_{cutoff}'] = val

        # Compute other metrics (without cutoff parameter)
        for name, func in metrics.METRIC_FUNCTIONS.items():
            if name in ['accL_cutoff', 'accP_cutoff']:
                continue  # Already computed above
            meta = func._metadata
            if meta['coord_convention'] == 'horizontal-polar':
                val, aux = func(targ_hp, resp_hp)
            elif meta['coord_convention'] == 'spherical':
                val, aux = func(targ_sph, resp_sph)
            unit = meta.get('output_unit', '')
            if unit == 'radians':
                row[f'{name}_deg'] = np.rad2deg(val)
            else:
                row[name] = val

        actual_metrics_rows.append(row)

    actual_behavioral_df = pd.DataFrame(actual_metrics_rows)

    # Save to CSV
    actual_behavioral_df.to_csv(behavioral_metrics_csv, index=False)
    print(f"✓ Saved behavioral metrics to {behavioral_metrics_csv.relative_to(paths.ROOT)}")

print("Actual behavioral metrics per participant:")
(actual_behavioral_df.round(2))

✓ Loading existing behavioral metrics from results/actual_behavioural_metrics.csv
Loaded 33 participants
Actual behavioral metrics per participant:


,participant,accL_cutoff_30_deg,accP_cutoff_30_deg,accL_cutoff_60_deg,accP_cutoff_60_deg,accL_cutoff_90_deg,accP_cutoff_90_deg,sdL_deg,rmsL_deg,rmsPmedianlocal_deg,querrMiddlebrooks,angular_error_deg
0,P0001,2.96,-14.30,4.63,-11.73,3.84,-6.24,11.24,10.96,34.49,7.21,412.94
1,P0006,-1.23,-10.16,-5.17,-8.00,-3.11,-9.01,11.18,10.46,37.03,11.11,636.62
2,P0019,2.96,1.47,-0.27,-1.73,0.78,-2.05,13.64,9.39,27.31,15.94,913.41
3,P0031,0.20,-2.09,0.82,-8.47,1.28,-8.50,10.81,9.42,36.82,7.35,421.29
4,P0043,0.03,1.30,-2.82,2.32,-3.05,3.49,9.25,8.88,20.80,9.52,545.67
5,P0124,-2.54,-0.10,1.59,1.42,3.06,1.84,15.58,15.73,40.99,12.96,742.72
6,P0172,-0.26,13.97,-0.12,13.31,-0.55,11.83,6.62,6.34,30.67,0.00,0.00
7,P0181,1.11,-9.14,1.76,-10.43,2.88,-13.03,8.64,7.25,30.78,12.50,716.20
8,P0015,1.46,15.05,-2.61,9.10,-0.73,4.97,10.13,8.03,36.49,6.35,363.78
9,P0107,-0.18,0.76,0.10,2.44,0.20,8.33,10.61,8.90,29.87,15.62,895.25


## Helper Functions

In [11]:
def compute_localization_metrics(responses_cart, targets_cart, target_indices, cutoffs=[30, 60, 90]):
    """
    Compute localization metrics using bayesian_listener.metrics module.

    Parameters
    ----------
    responses_cart : ndarray
        Response positions in Cartesian coordinates (n_obs x 3)
    targets_cart : ndarray
        All target positions in Cartesian coordinates (n_targets x 3)
    target_indices : ndarray
        Index mapping each response to a target (n_obs,)
    cutoffs : list
        Cutoff angles in degrees for accL_cutoff and accP_cutoff

    Returns
    -------
    metrics_dict : dict
        Dictionary containing various localization metrics
    """
    # Get target positions for each response
    response_targets = targets_cart[target_indices, :]

    # Convert to horizontal-polar (lateral, polar) in radians using pyfar
    resp_coords = pf.Coordinates.from_cartesian(
        responses_cart[:, 0], responses_cart[:, 1], responses_cart[:, 2])
    resp_hp = resp_coords.spherical_side  # (n_obs, 3) radians

    targ_coords = pf.Coordinates.from_cartesian(
        response_targets[:, 0], response_targets[:, 1], response_targets[:, 2])
    targ_hp = targ_coords.spherical_side  # (n_obs, 3) radians

    # Compute basic metrics
    rmsL_val, _ = metrics.METRIC_FUNCTIONS['rmsL'](targ_hp, resp_hp)
    rmsPmedianlocal_val, _ = metrics.METRIC_FUNCTIONS['rmsPmedianlocal'](targ_hp, resp_hp)
    querr_val, querr_aux = metrics.METRIC_FUNCTIONS['querrMiddlebrooks'](targ_hp, resp_hp)

    result = {
        'rmsL': rmsL_val,
        'rmsL_deg': np.rad2deg(rmsL_val),
        'rmsPmedianlocal': rmsPmedianlocal_val,
        'rmsPmedianlocal_deg': np.rad2deg(rmsPmedianlocal_val),
        'querrMiddlebrooks': querr_val,
        'querr_confusion_count': querr_aux['confusion_count'],
        'querr_response_count': querr_aux['response_count']
    }

    # Compute cutoff metrics at different angles (cutoffs in degrees, passed as radians)
    for cutoff in cutoffs:
        accL_val, _ = metrics.METRIC_FUNCTIONS['accL_cutoff'](
            targ_hp, resp_hp, cutoff=np.deg2rad(cutoff))
        accP_val, _ = metrics.METRIC_FUNCTIONS['accP_cutoff'](
            targ_hp, resp_hp, cutoff=np.deg2rad(cutoff))
        result[f'accL_cutoff_{cutoff}_deg'] = np.rad2deg(accL_val)
        result[f'accP_cutoff_{cutoff}_deg'] = np.rad2deg(accP_val)

    return result

print("✓ Helper functions defined")

✓ Helper functions defined


## Generate Synthetic Data

For each participant and method, generate 10 synthetic responses per target using the fitted parameters.

In [12]:
def generate_synthetic_responses(participant_id, method, params, model, targets, n_reps=200, seed=None):
    """
    Generate synthetic localization responses using fitted parameters.
    Uses the model's estimate() method to apply motor noise.

    Parameters
    ----------
    participant_id : str
        participant identifier
    method : str
        Interpolation method
    params : dict
        Fitted parameters containing sigma_ild, sigma_spectral, kappa_motor, sigma_prior
    model : BayesianListener
        Model instance with template already computed
    targets : AuditoryRepresentation
        Target feature subset (n_targets directions)
    n_reps : int
        Number of repetitions per target

    Returns
    -------
    responses_cart : ndarray
        Synthetic responses in Cartesian coordinates (n_targets * n_reps x 3)
    target_indices : ndarray
        Index mapping each response to a target (n_targets * n_reps,)
    """
    # Set model parameters
    model.parameters = {
        'sigma_itd': 0.569,  # Fixed ITD noise
        'sigma_ild': params['sigma_ild'],
        'sigma_spectral': params['sigma_spectral'],
        'sigma_prior': params['sigma_prior'],
        'kappa_motor': params['kappa_motor']
    }

    # Set target features on the model and run inference
    model.target = targets
    posterior = model.infer(repetitions=n_reps, seed=seed)

    # Get MAP estimates with motor noise; estimate() returns pyfar.Coordinates
    # Shape: cshape (n_targets, n_reps), cartesian (n_targets, n_reps, 3)
    doa_estimations = model.estimate(posterior, kappa_motor=params['kappa_motor'])
    doa_cart = doa_estimations.cartesian  # (n_targets, n_reps, 3)

    n_targets = doa_cart.shape[0]
    responses_cart = doa_cart.reshape(-1, 3)
    target_indices = np.repeat(np.arange(n_targets), n_reps)

    return responses_cart, target_indices

print("✓ Synthetic data generation function defined")

✓ Synthetic data generation function defined


## Compute Metrics for All participants and Methods

In [13]:
# Compute Metrics for All participants and Methods (Parallelized)
# Skip if CSV already exists
from joblib import Parallel, delayed, cpu_count

synthetic_vs_actual_csv = paths.RESULTS_DIR / 'synthetic_vs_actual_comparison.csv'

if Path(synthetic_vs_actual_csv).exists():
    print(f"✓ Loading existing synthetic vs actual comparison from {synthetic_vs_actual_csv.relative_to(paths.ROOT)}")
    metrics_df = pd.read_csv(synthetic_vs_actual_csv)
    print(f"Loaded {len(metrics_df)} metric combinations")
else:
    print(f"Computing synthetic vs actual metrics (CSV not found)...")

    # Extract Cartesian array from pyfar for use in compute_localization_metrics
    targets_cart = targets_coords.cartesian

    print("Computing metrics for all participants and methods (parallel)...\n")

    methods = ['SHMAX', 'SH', 'barycentric', 'barumerli2023']

    def process_participant_actual(participant_id, obs_tbl, targets_coords, targets_cart):
        """Compute actual metrics for a single participant."""
        try:
            subj_data = obs_tbl[obs_tbl['participant'] == participant_id]

            # Build pyfar response coordinates and convert to Cartesian
            actual_resp_coords = pf.Coordinates.from_spherical_elevation(
                np.deg2rad(subj_data['azi_response'].values),
                np.deg2rad(subj_data['ele_response'].values),
                np.ones(len(subj_data))
            )
            actual_resp_cart = actual_resp_coords.cartesian

            # Get target indices for actual responses
            actual_resp_targets = pf.Coordinates.from_spherical_elevation(
                np.deg2rad(subj_data['azi_target'].values),
                np.deg2rad(subj_data['ele_target'].values),
                np.ones(len(subj_data))
            )
            actual_target_indices = targets_coords.find_nearest(actual_resp_targets)[0][0]

            actual_metrics = compute_localization_metrics(
                actual_resp_cart, targets_cart, actual_target_indices
            )
            actual_metrics['participant'] = participant_id
            actual_metrics['method'] = 'actual'
            actual_metrics['data_type'] = 'actual'

            return actual_metrics
        except Exception as e:
            print(f"Error processing actual data for {participant_id}: {str(e)}")
            return None

    def process_participant_method_synthetic(participant_id, method, fitted_params_df,
                                            data_folder, targets_coords, targets_cart):
        """Compute synthetic metrics for a single participant-method combination."""
        try:
            param_row = fitted_params_df[
                (fitted_params_df['participant'] == participant_id) &
                (fitted_params_df['method'] == method)
            ]

            if len(param_row) == 0:
                print(f"Warning: No fitted parameters for {participant_id} - {method}")
                return None

            params = {
                'sigma_ild': param_row['sigma_ild'].values[0],
                'sigma_spectral': param_row['sigma_spectral'].values[0],
                'kappa_motor': param_row['kappa_motor'].values[0],
                'sigma_prior': param_row['sigma_prior'].values[0]
            }

            # Load HRTF and compute template
            subj_sofa_path = f'{data_folder}/{participant_id}_FreeFieldCompMinPhase_48kHz.sofa'
            model = BayesianListener(subj_sofa_path)
            model.compute_template(interpolation=method)

            # Extract target feature subset from model
            target_indices_in_model = model.target.coords.find_nearest(targets_coords)[0][0]
            targets = model.target[target_indices_in_model]

            # Generate synthetic responses (seed derived from participant + method for reproducibility)
            _seed = hash((participant_id, method)) % (2**31)
            synth_resp_cart, synth_target_indices = generate_synthetic_responses(
                participant_id, method, params, model, targets, seed=_seed
            )

            synth_metrics = compute_localization_metrics(
                synth_resp_cart, targets_cart, synth_target_indices
            )
            synth_metrics['participant'] = participant_id
            synth_metrics['method'] = method
            synth_metrics['data_type'] = 'synthetic'

            return synth_metrics
        except Exception as e:
            print(f"Error processing {participant_id} - {method}: {str(e)}")
            return None

    # Process actual metrics in parallel
    print("Processing actual behavioral data...")
    actual_results = Parallel(n_jobs=cpu_count(), verbose=5)(
        delayed(process_participant_actual)(pid, obs_tbl, targets_coords, targets_cart)
        for pid in participants
    )

    actual_metrics_list = [r for r in actual_results if r is not None]

    # Process synthetic metrics in parallel
    print("\nProcessing synthetic data for all methods...")
    synthetic_tasks = [
        (pid, method) for pid in participants for method in methods
    ]

    synthetic_results = Parallel(n_jobs=cpu_count(), verbose=5)(
        delayed(process_participant_method_synthetic)(
            pid, method, fitted_params_df, data_folder, targets_coords, targets_cart
        )
        for pid, method in synthetic_tasks
    )

    synthetic_metrics_list = [r for r in synthetic_results if r is not None]

    all_metrics = actual_metrics_list + synthetic_metrics_list
    metrics_df = pd.DataFrame(all_metrics)

    metrics_df.to_csv(synthetic_vs_actual_csv, index=False)
    print(f"\n✓ Saved synthetic vs actual comparison to {synthetic_vs_actual_csv.relative_to(paths.ROOT)}")

print(f"\n✓ Computed metrics for {len(metrics_df)} combinations")

✓ Loading existing synthetic vs actual comparison from results/synthetic_vs_actual_comparison.csv
Loaded 165 metric combinations

✓ Computed metrics for 165 combinations


## Individual-Level Comparison

Compare actual vs. synthetic metrics for each participant and method.

In [14]:
# Prepare data for comparison
actual_df = metrics_df[metrics_df['data_type'] == 'actual'].copy()
synth_df = metrics_df[metrics_df['data_type'] == 'synthetic'].copy()


In [15]:
# Merge actual and synthetic for each participant-method combination
methods = ['SHMAX', 'SH', 'barycentric', 'barumerli2023']
comparison_rows = []

for participant in participants:
    actual_row = actual_df[actual_df['participant'] == participant].iloc[0]

    for method in methods:
        synth_row = synth_df[
            (synth_df['participant'] == participant) &
            (synth_df['method'] == method)
        ]

        if len(synth_row) == 0:
            continue
        synth_row = synth_row.iloc[0]

        row = {
            'participant': participant,
            'method': method,
            'actual_rmsL': actual_row['rmsL_deg'],
            'synth_rmsL': synth_row['rmsL_deg'],
            'diff_rmsL': synth_row['rmsL_deg'] - actual_row['rmsL_deg'],
            'actual_rmsPlocal': actual_row['rmsPmedianlocal_deg'],
            'synth_rmsPlocal': synth_row['rmsPmedianlocal_deg'],
            'diff_rmsPlocal': synth_row['rmsPmedianlocal_deg'] - actual_row['rmsPmedianlocal_deg'],
            'actual_querr': actual_row['querrMiddlebrooks'],
            'synth_querr': synth_row['querrMiddlebrooks'],
            'diff_querr': synth_row['querrMiddlebrooks'] - actual_row['querrMiddlebrooks']
        }

        # Add accL_cutoff metrics for all cutoffs
        for cutoff in [30, 60, 90]:
            col_name = f'accL_cutoff_{cutoff}_deg'
            row[f'actual_accL_{cutoff}'] = actual_row[col_name]
            row[f'synth_accL_{cutoff}'] = synth_row[col_name]
            row[f'diff_accL_{cutoff}'] = synth_row[col_name] - actual_row[col_name]

        comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)

print("Individual-Level Comparison Summary:")
print("="*80)
print(comparison_df.to_string(index=False))

Individual-Level Comparison Summary:
participant        method  actual_rmsL  synth_rmsL  diff_rmsL  actual_rmsPlocal  synth_rmsPlocal  diff_rmsPlocal  actual_querr  synth_querr  diff_querr  actual_accL_30  synth_accL_30  diff_accL_30  actual_accL_60  synth_accL_60  diff_accL_60  actual_accL_90  synth_accL_90  diff_accL_90
      P0001         SHMAX    10.959092   13.044823   2.085732         34.490431        28.434599       -6.055833      7.207207     8.715925    1.508718        3.837133      -0.180215     -4.017348        3.837133      -0.180215     -4.017348        3.837133      -0.180215     -4.017348
      P0001            SH    10.959092   13.140439   2.181347         34.490431        26.654964       -7.835468      7.207207     6.928645   -0.278562        3.837133      -0.293543     -4.130676        3.837133      -0.293543     -4.130676        3.837133      -0.293543     -4.130676
      P0001   barycentric    10.959092   13.075594   2.116502         34.490431        29.773782      

In [16]:
# Table 1: Behavioral Metrics Summary with Standard Errors and Significance
# Methods as rows, Metrics as columns (rmsL, rmsP, querr only)
# Format: mean ± SE (italic if not significantly different from actual)

methods = ['SHMAX', 'SH', 'barycentric', 'barumerli2023']
metric_names = ['rmsL', 'rmsP', 'querr']
metric_labels = {
    'rmsL': 'rmsL (°)',
    'rmsP': 'rmsP (°)',
    'querr': 'querr (\\%)'
}

# Compute actual metrics with SE
actual_metrics = {
    'rmsL': (actual_df['rmsL_deg'].mean(), actual_df['rmsL_deg'].sem()),
    'rmsP': (actual_df['rmsPmedianlocal_deg'].mean(), actual_df['rmsPmedianlocal_deg'].sem()),
    'querr': (actual_df['querrMiddlebrooks'].mean(), actual_df['querrMiddlebrooks'].sem())
}

# Collect synthetic data and perform t-tests for each method
method_data = {}
for method in methods:
    method_data[method] = {
        'values': {'rmsL': [], 'rmsP': [], 'querr': []},
        'actual': {'rmsL': [], 'rmsP': [], 'querr': []},
        'p_values': {}
    }

    for participant in actual_df['participant'].unique():
        actual_p = actual_df[actual_df['participant'] == participant].iloc[0]
        synth_p = synth_df[(synth_df['participant'] == participant) &
                           (synth_df['method'] == method)]

        if len(synth_p) > 0:
            synth_p = synth_p.iloc[0]

            method_data[method]['actual']['rmsL'].append(actual_p['rmsL_deg'])
            method_data[method]['values']['rmsL'].append(synth_p['rmsL_deg'])

            method_data[method]['actual']['rmsP'].append(actual_p['rmsPmedianlocal_deg'])
            method_data[method]['values']['rmsP'].append(synth_p['rmsPmedianlocal_deg'])

            method_data[method]['actual']['querr'].append(actual_p['querrMiddlebrooks'])
            method_data[method]['values']['querr'].append(synth_p['querrMiddlebrooks'])

    # Perform paired t-tests for each metric
    for metric in metric_names:
        _, p_val = stats.ttest_rel(
            method_data[method]['actual'][metric],
            method_data[method]['values'][metric]
        )
        method_data[method]['p_values'][metric] = p_val

# Build table with methods as rows
table1_data = []

# Actual row
actual_row = {'Method': 'Actual'}
for metric in metric_names:
    actual_row[metric_labels[metric]] = f"{actual_metrics[metric][0]:.1f} $\\pm$ {actual_metrics[metric][1]:.1f}"
table1_data.append(actual_row)

# Synthetic rows for each method
for method in methods:
    row = {'Method': method}

    for metric in metric_names:
        vals = method_data[method]['values'][metric]
        mean = np.mean(vals)
        se = np.std(vals) / np.sqrt(len(vals))
        p_val = method_data[method]['p_values'][metric]

        value_str = f"{mean:.1f} $\\pm$ {se:.1f}"
        if p_val < 0.05:
            # Non-significant = good fit, use italics
            row[metric_labels[metric]] = f"\\textit{{{value_str}}}"
        else:
            # Significant difference = poor fit
            row[metric_labels[metric]] = value_str

    table1_data.append(row)

table1_df = pd.DataFrame(table1_data)
print("Table 1: Behavioral Metrics Summary (mean ± SE)")
print("Italic values indicate no significant difference from actual (p > 0.05)")
print("="*80)
print(table1_df)
print("\nLaTeX:")
print(table1_df.to_latex(index=False, escape=False))


# Table 2: Correlation and P-values
# Methods as rows, Metrics as columns

table2_data = []

for method in methods:
    row = {'Method': method}

    for metric in metric_names:
        # Get actual and synthetic values
        actual_vals = method_data[method]['actual'][metric]
        synth_vals = method_data[method]['values'][metric]

        # Compute correlation
        r, p = stats.pearsonr(actual_vals, synth_vals)

        # Format with asterisks for significance
        if p < 0.001:
            row[metric_labels[metric]] = f"{r:.2f}***"
        elif p < 0.01:
            row[metric_labels[metric]] = f"{r:.2f}**"
        elif p < 0.05:
            row[metric_labels[metric]] = f"{r:.2f}*"
        else:
            row[metric_labels[metric]] = f"{r:.2f}"

    table2_data.append(row)

table2_df = pd.DataFrame(table2_data)

print("\n" + "="*80)
print("Table 2: Pearson Correlations (r) between Actual and Synthetic")
print("* p < 0.05, ** p < 0.01, *** p < 0.001")
print("="*80)
print(table2_df)
print("\nLaTeX:")
print(table2_df.to_latex(index=False, escape=False))

Table 1: Behavioral Metrics Summary (mean ± SE)
Italic values indicate no significant difference from actual (p > 0.05)
          Method        rmsL (°)        rmsP (°)      querr (\%)
0         Actual  12.7 $\pm$ 0.9  33.8 $\pm$ 1.1  15.1 $\pm$ 1.8
1          SHMAX  13.3 $\pm$ 1.2  33.7 $\pm$ 0.8  16.4 $\pm$ 1.3
2             SH  13.5 $\pm$ 1.2  33.9 $\pm$ 0.8  16.4 $\pm$ 1.1
3    barycentric  13.3 $\pm$ 1.2  33.4 $\pm$ 0.8  15.5 $\pm$ 1.1
4  barumerli2023  13.6 $\pm$ 1.2  34.3 $\pm$ 0.7  17.4 $\pm$ 1.1

LaTeX:
\begin{tabular}{llll}
\toprule
Method & rmsL (°) & rmsP (°) & querr (\%) \\
\midrule
Actual & 12.7 $\pm$ 0.9 & 33.8 $\pm$ 1.1 & 15.1 $\pm$ 1.8 \\
SHMAX & 13.3 $\pm$ 1.2 & 33.7 $\pm$ 0.8 & 16.4 $\pm$ 1.3 \\
SH & 13.5 $\pm$ 1.2 & 33.9 $\pm$ 0.8 & 16.4 $\pm$ 1.1 \\
barycentric & 13.3 $\pm$ 1.2 & 33.4 $\pm$ 0.8 & 15.5 $\pm$ 1.1 \\
barumerli2023 & 13.6 $\pm$ 1.2 & 34.3 $\pm$ 0.7 & 17.4 $\pm$ 1.1 \\
\bottomrule
\end{tabular}


Table 2: Pearson Correlations (r) between Actual and Synt

## Individual-Level Visualization

Fig. 5 (actual vs simulated LE, PE and QE with regression lines) is drawn from
`results/synthetic_vs_actual_comparison.csv` by `figures/fig_metrics_actual_vs_synthetic.py`.

## Group-Level Comparison

In [17]:
# Group statistics
group_stats = comparison_df.groupby('method').agg({
    'actual_rmsL': ['mean', 'std'],
    'synth_rmsL': ['mean', 'std'],
    'diff_rmsL': ['mean', 'std'],
    'actual_rmsPlocal': ['mean', 'std'],
    'synth_rmsPlocal': ['mean', 'std'],
    'diff_rmsPlocal': ['mean', 'std'],
    'actual_querr': ['mean', 'std'],
    'synth_querr': ['mean', 'std'],
    'diff_querr': ['mean', 'std']
}).round(2)

print("Group-Level Statistics:")
print("="*80)
print(group_stats.T)

Group-Level Statistics:
method                    SH  SHMAX  barumerli2023  barycentric
actual_rmsL      mean  12.72  12.72          12.72        12.72
                 std    5.19   5.19           5.19         5.19
synth_rmsL       mean  13.45  13.31          13.62        13.32
                 std    7.16   7.14           6.89         7.16
diff_rmsL        mean   0.74   0.59           0.90         0.61
                 std    3.45   3.42           3.27         3.42
actual_rmsPlocal mean  33.79  33.79          33.79        33.79
                 std    6.24   6.24           6.24         6.24
synth_rmsPlocal  mean  33.93  33.73          34.30        33.43
                 std    4.84   4.81           4.35         4.79
diff_rmsPlocal   mean   0.14  -0.05           0.51        -0.36
                 std    5.30   4.90           4.75         4.81
actual_querr     mean  15.09  15.09          15.09        15.09
                 std   10.62  10.62          10.62        10.62
synth_querr     

## Statistical Tests: Actual vs. Synthetic

In [18]:
# Paired t-tests for each method
print("Paired t-tests: Actual vs. Synthetic Lateral RMS Error")
print("="*80)

for method in methods:
    method_data = comparison_df[comparison_df['method'] == method]

    t_stat, p_val = stats.ttest_rel(
        method_data['actual_rmsL'],
        method_data['synth_rmsL']
    )

    mean_diff = method_data['diff_rmsL'].mean()

    print(f"{method:15s}: t={t_stat:6.3f}, p={p_val:.4f}, mean_diff={mean_diff:6.2f}°")

print("\nPaired t-tests: Actual vs. Synthetic Local Polar RMS Error")
print("="*80)

for method in methods:
    method_data = comparison_df[comparison_df['method'] == method]

    t_stat, p_val = stats.ttest_rel(
        method_data['actual_rmsPlocal'],
        method_data['synth_rmsPlocal']
    )

    mean_diff = method_data['diff_rmsPlocal'].mean()

    print(f"{method:15s}: t={t_stat:6.3f}, p={p_val:.4f}, mean_diff={mean_diff:6.2f}°")

print("\nPaired t-tests: Actual vs. Synthetic Querr")
print("="*80)

for method in methods:
    method_data = comparison_df[comparison_df['method'] == method]

    t_stat, p_val = stats.ttest_rel(
        method_data['actual_querr'],
        method_data['synth_querr']
    )

    mean_diff = method_data['diff_querr'].mean()

    print(f"{method:15s}: t={t_stat:6.3f}, p={p_val:.4f}, mean_diff={mean_diff:6.2f}°")

Paired t-tests: Actual vs. Synthetic Lateral RMS Error
SHMAX          : t=-0.991, p=0.3292, mean_diff=  0.59°
SH             : t=-1.229, p=0.2281, mean_diff=  0.74°
barycentric    : t=-1.021, p=0.3149, mean_diff=  0.61°
barumerli2023  : t=-1.587, p=0.1222, mean_diff=  0.90°

Paired t-tests: Actual vs. Synthetic Local Polar RMS Error
SHMAX          : t= 0.063, p=0.9500, mean_diff= -0.05°
SH             : t=-0.154, p=0.8786, mean_diff=  0.14°
barycentric    : t= 0.425, p=0.6735, mean_diff= -0.36°
barumerli2023  : t=-0.620, p=0.5399, mean_diff=  0.51°

Paired t-tests: Actual vs. Synthetic Querr
SHMAX          : t=-0.822, p=0.4174, mean_diff=  1.32°
SH             : t=-0.794, p=0.4328, mean_diff=  1.27°
barycentric    : t=-0.311, p=0.7580, mean_diff=  0.46°
barumerli2023  : t=-1.170, p=0.2505, mean_diff=  2.30°


## Method Comparison: Which Method Best Predicts Actual Data?

In [19]:
# Compute absolute differences for each method
print("Mean Absolute Difference from Actual Data:")
print("="*80)

method_comparison = []

for method in methods:
    method_data = comparison_df[comparison_df['method'] == method]

    comp_dict = {
        'method': method,
        'MAD_rmsL': np.mean(np.abs(method_data['diff_rmsL'])),
        'MAD_rmsPlocal': np.mean(np.abs(method_data['diff_rmsPlocal'])),
        'MAD_querr': np.mean(np.abs(method_data['diff_querr']))
    }

    # Add MAD for accL_cutoff at all cutoffs
    for cutoff in [30, 60, 90]:
        comp_dict[f'MAD_accL_{cutoff}'] = np.mean(np.abs(method_data[f'diff_accL_{cutoff}']))

    method_comparison.append(comp_dict)

method_comp_df = pd.DataFrame(method_comparison)
print(method_comp_df.round(2))

# Find best method for each metric
print("\nBest Method (lowest MAD):")
metric_cols = ['MAD_rmsL', 'MAD_rmsPlocal', 'MAD_querr', 'MAD_accL_30', 'MAD_accL_60', 'MAD_accL_90']
for col in metric_cols:
    best_method = method_comp_df.loc[method_comp_df[col].idxmin(), 'method']
    best_value = method_comp_df[col].min()
    metric_label = col.replace('MAD_', '')
    if 'querr' in col:
        print(f"  {col:20s}: {best_method:15s} (MAD = {best_value:.2f}%)")
    else:
        print(f"  {col:20s}: {best_method:15s} (MAD = {best_value:.2f}°)")

Mean Absolute Difference from Actual Data:
          method  MAD_rmsL  MAD_rmsPlocal  MAD_querr  MAD_accL_30  \
0          SHMAX      2.58           4.11       6.70         5.50   
1             SH      2.56           4.29       7.08         5.48   
2    barycentric      2.57           3.78       6.23         5.52   
3  barumerli2023      2.53           3.88       8.92         5.48   

   MAD_accL_60  MAD_accL_90  
0         5.50         5.50  
1         5.48         5.48  
2         5.52         5.52  
3         5.48         5.48  

Best Method (lowest MAD):
  MAD_rmsL            : barumerli2023   (MAD = 2.53°)
  MAD_rmsPlocal       : barycentric     (MAD = 3.78°)
  MAD_querr           : barycentric     (MAD = 6.23%)
  MAD_accL_30         : SH              (MAD = 5.48°)
  MAD_accL_60         : SH              (MAD = 5.48°)
  MAD_accL_90         : SH              (MAD = 5.48°)
